# Задача 7. Attribution != Incrementality

После рекламного поста продажи выросли — но выросли ли они ИЗ-ЗА него,
или выросли бы всё равно?

Код вынесен в [`src/incrementality.py`](../src/incrementality.py) —
ноутбук его только вызывает и объясняет, что означают цифры.

Полный текстовый разбор — в
[`reports/TASK7_INCREMENTALITY.md`](../reports/TASK7_INCREMENTALITY.md).

In [1]:
import sys
sys.path.append("../src")
from incrementality import (
    load_daily_orders, diff_in_diff, weekend_confound_range,
    required_sample_size, power_simulation, assign_group,
)

orders = load_daily_orders("../data/base.xlsx")
orders.head()

,student_id,ts,courses,date,is_pro,is_start
0,1,2026-09-04 19:30:57,[Аналитика старт],2026-09-04,False,True
1,2,2026-09-05 16:22:48,[ML про],2026-09-05,True,False
2,3,2026-09-03 22:47:02,[АВ тестам],2026-09-03,False,False
3,4,2026-08-31 15:42:34,[AI агенты],2026-08-31,False,False
4,5,2026-08-22 16:19:00,[ML про],2026-08-22,True,False


## 1. Естественный эксперимент: продвигали ПРО, СТАРТ не трогали

20-23 августа шло продвижение линейки ПРО. Линейку СТАРТ в это время не
рекламировали — она играет роль контрольной группы. Сравниваем не
"было / стало" у ПРО (это спутало бы эффект рекламы с общим ростом
продаж к концу августа), а разницу приростов у ПРО и у СТАРТ за один
и тот же период.

In [2]:
did = diff_in_diff(
    orders,
    pre_start="2026-08-15", pre_end="2026-08-19",
    treat_start="2026-08-20", treat_end="2026-08-23",
)

print(f'ПРО: было {did["pro_pre"]:.2f} заказов/день -> стало {did["pro_treat"]:.2f} '
      f'(прирост {did["pro_delta"]:+.2f})')
print(f'СТАРТ (контроль): было {did["start_pre"]:.2f} -> стало {did["start_treat"]:.2f} '
      f'(прирост {did["start_delta"]:+.2f})')
print(f'\nЭффект рекламы = разница приростов = {did["effect_per_day"]:+.2f} заказов/день')

ПРО: было 0.80 заказов/день -> стало 10.25 (прирост +9.45)
СТАРТ (контроль): было 3.20 -> стало 4.00 (прирост +0.80)

Эффект рекламы = разница приростов = +8.65 заказов/день


**Вывод.** У ПРО прирост почти в 12 раз больше, чем у СТАРТ за то же
время. Разница приростов — около **+8,6 заказа в день** — это и есть
оценка эффекта рекламы.

## 2. Всплеск 9 августа: скидка и выходной день неразличимы

9 августа было воскресенье, и в этот же день шла крупная распродажа.
Разделить два эффекта (выходной vs скидка) на этих данных нельзя.
Честный ответ — вилка, а не одно число.

In [3]:
rng = weekend_confound_range(orders, spike_date="2026-08-09")

print(f'Заказов 9 августа: {rng["spike_orders"]:.0f}')
print(f'Если бы день вёл себя как обычный выходной, лишних заказов: {rng["lower_bound"]:.0f}')
print(f'Если бы день вёл себя как обычный будний день, лишних заказов: {rng["upper_bound"]:.0f}')
print(f'\nЧестный ответ: реклама дала от {rng["lower_bound"]:.0f} до {rng["upper_bound"]:.0f} '
      f'дополнительных заказов — точнее сказать на этих данных нельзя.')

Заказов 9 августа: 68
Если бы день вёл себя как обычный выходной, лишних заказов: 42
Если бы день вёл себя как обычный будний день, лишних заказов: 56

Честный ответ: реклама дала от 42 до 56 дополнительных заказов — точнее сказать на этих данных нельзя.


## 4. Holdout в рассылке бота — дизайн для будущего

Единственное место, где можно по-честному разделить пользователей на
"показали рекламу" и "не показали" — это рассылка бота. Считаем, сколько
человек нужно на группу, чтобы поймать заданный эффект.

In [4]:
n = required_sample_size(baseline_rate=0.05, relative_lift=0.5)
print(f"Нужно человек на группу (баланс 5%, ищем лифт +50%): {n}")

# Пример детерминированного назначения в группу
for uid in ["user_1", "user_2", "user_3"]:
    print(uid, "->", assign_group(uid, experiment_name="autumn_holdout"))

Нужно человек на группу (баланс 5%, ищем лифт +50%): 1468
user_1 -> treatment
user_2 -> control
user_3 -> treatment


**Почему хэш, а не случайный рандом при каждом запуске.** Один и тот же
человек должен каждый раз попадать в одну и ту же группу — иначе тест
не воспроизводим и его нельзя перепроверить.

## 5. Честная проверка: когда тесту вообще можно доверять

Симулируем: если реальный эффект рекламы +30%, как часто тест разного
размера его заметит?

In [5]:
for n_per_group in (600, 5_000, 40_000):
    p = power_simulation(n_per_group, baseline_rate=0.05, relative_lift=0.30)
    print(f"n={n_per_group:>6}: эффект +30% обнаружен в {p:.1%} симуляций")

n=   600: эффект +30% обнаружен в 19.8% симуляций
n=  5000: эффект +30% обнаружен в 89.7% симуляций
n= 40000: эффект +30% обнаружен в 100.0% симуляций


**Вывод.** При маленьком холдауте (600 человек) тест почти всегда
пропустит реальный эффект +30% — не потому что рекламы нет, а потому что
выборки не хватает. При 40 000 — обнаруживает уверенно.